# ANFIS Predict — Toàn bộ tập 663 mẫu

Load model **grid search** tốt nhất (70/30) và dự đoán trên **toàn bộ 663 mẫu** sau tiền xử lý — cùng pipeline với `anfis_wbcd_pretrain_baseline.ipynb` và `anfis_pca_scikit_anfis_training.ipynb`.

1. Loại missing → loại outlier → chuẩn hóa bằng `scaler.pkl` đã lưu → chọn 3 feature paper → quality drop còn **663 mẫu**
2. Khởi tạo FS + 27 luật grid, load checkpoint
3. Predict toàn bộ 663 mẫu và lưu kết quả

In [16]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

from skanfis import scikit_anfis
from skanfis.fs import FS, LinguisticVariable, GaussianFuzzySet

print("Torch:", torch.__version__)

Torch: 2.12.1+cpu


In [17]:
# Cau hinh artifact grid search — model tot nhat split 70/30
STAMP = "20260628_172814"
RUN_TAG = "gridsearch_pca_feature_select"
SPLIT_KEY = "70-30"
SPLIT_TAG = "70_30"

models_dir = Path("models")
model_path = models_dir / f"{STAMP}_{RUN_TAG}_{SPLIT_TAG}_best_model.pkl"
scaler_path = models_dir / f"{STAMP}_{RUN_TAG}_scaler.pkl"
meta_path = models_dir / f"{STAMP}_{RUN_TAG}_meta.json"

for p in [model_path, scaler_path, meta_path]:
    if not p.exists():
        raise FileNotFoundError(f"Thieu file: {p}")

with open(meta_path, encoding="utf-8") as f:
    meta = json.load(f)

with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

split_cfg = meta["splits"][SPLIT_KEY]
selected_feature_names = meta["selected_features"]
USED_SAMPLES = meta["used_samples"]

print("Model:", model_path.name)
print("Selected features:", selected_feature_names)
print("Danh gia tren:", USED_SAMPLES, "mau (sau outlier + quality drop)")

Model: 20260628_172814_gridsearch_pca_feature_select_70_30_best_model.pkl
Selected features: ['clump_thickness', 'uniformity_of_cell_size', 'uniformity_of_cell_shape']
Danh gia tren: 663 mau (sau outlier + quality drop)


In [18]:
# Load WBCD goc (UCI Breast Cancer Wisconsin Original)
uci_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

cols = [
    "sample_code_number",
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
    "class",
]

df = pd.read_csv(uci_url, header=None, names=cols)
df = df.replace("?", np.nan).dropna().copy()
df["bare_nuclei"] = df["bare_nuclei"].astype(int)
df["target"] = (df["class"] == 4).astype(int)
df["target_label"] = df["target"].map({0: "benign", 1: "malignant"})

feature_cols = [
    "clump_thickness",
    "uniformity_of_cell_size",
    "uniformity_of_cell_shape",
    "marginal_adhesion",
    "single_epithelial_cell_size",
    "bare_nuclei",
    "bland_chromatin",
    "normal_nucleoli",
    "mitoses",
]

X_raw = df[feature_cols].values.astype(np.float32)
y_true = df["target"].values.astype(int)

print("WBCD sau loai missing:", X_raw.shape)
print("Class balance (malignant rate):", round(float(y_true.mean()), 4))

WBCD sau loai missing: (683, 9)
Class balance (malignant rate): 0.3499


In [19]:
# Tien xu ly giong training + pretrain baseline: outlier -> scaler -> 3 feature -> quality drop -> 663
OUTLIER_STD_MULTIPLIER = meta.get("outlier_std_multiplier", 2.5)

data_center = X_raw.mean(axis=0)
euclidean_distances = np.linalg.norm(X_raw - data_center, axis=1)
outlier_threshold = euclidean_distances.mean() + OUTLIER_STD_MULTIPLIER * euclidean_distances.std()
keep_mask = euclidean_distances <= outlier_threshold

df_clean = df.iloc[keep_mask].reset_index(drop=True)
X_clean = df_clean[feature_cols].values.astype(np.float32)
y_clean = df_clean["target"].values.astype(int)

print(f"Outlier loai bo: {int((~keep_mask).sum())} mau, con lai: {len(X_clean)}")

X_norm = scaler.transform(X_clean).astype(np.float32)
selected_idx = [feature_cols.index(c) for c in selected_feature_names]
X_selected = X_norm[:, selected_idx].astype(np.float32)

n_drop = len(X_selected) - USED_SAMPLES
centroid_benign = X_selected[y_clean == 0].mean(axis=0)
centroid_malignant = X_selected[y_clean == 1].mean(axis=0)
dist_to_benign = np.linalg.norm(X_selected - centroid_benign, axis=1)
dist_to_malignant = np.linalg.norm(X_selected - centroid_malignant, axis=1)
own_dist = np.where(y_clean == 0, dist_to_benign, dist_to_malignant)
other_dist = np.where(y_clean == 0, dist_to_malignant, dist_to_benign)
quality_score = other_dist - own_dist

drop_idx = np.argsort(quality_score)[:n_drop]
quality_keep_mask = np.ones(len(X_selected), dtype=bool)
quality_keep_mask[drop_idx] = False

df_663 = df_clean.iloc[quality_keep_mask].reset_index(drop=True)
X_model = X_selected[quality_keep_mask]
y_true = df_663["target"].values.astype(int)

# Split 70/30 — chi de khoi tao MF giong luc train (giong pretrain_baseline)
idx_all = np.arange(len(X_model))
idx_train, _ = train_test_split(
    idx_all,
    test_size=split_cfg["test_size"],
    random_state=42,
    stratify=y_true,
)
X_train = X_model[idx_train]

assert len(X_model) == USED_SAMPLES, f"Expected {USED_SAMPLES} samples, got {len(X_model)}"
print("Input shape cho ANFIS:", X_model.shape)
print("MF init tu tap train:", X_train.shape)
print("Malignant rate (663):", round(float(y_true.mean()), 4))

Outlier loai bo: 18 mau, con lai: 665
Input shape cho ANFIS: (663, 3)
MF init tu tap train: (464, 3)
Malignant rate (663): 0.3333


In [20]:
# Khoi tao FS tu X_train + load checkpoint (giong anfis_wbcd_pretrain_baseline.ipynb)
FS_VAR_NAMES = {
    "clump_thickness": "ClumpThickness",
    "uniformity_of_cell_size": "CellSize",
    "uniformity_of_cell_shape": "CellShape",
}
feature_name_map = {
    "clump_thickness": "Clump Thickness",
    "uniformity_of_cell_size": "Uniformity of Cell Size",
    "uniformity_of_cell_shape": "Uniformity of Cell Shape",
}
FS_DISPLAY_NAMES = {v: feature_name_map[k] for k, v in FS_VAR_NAMES.items()}
LINGUISTIC_TERMS = ("low", "medium", "high")


def build_fs_and_model(X_tr, epochs):
    fs = FS()
    for feat_col in selected_feature_names:
        fs_var = FS_VAR_NAMES[feat_col]
        col = X_tr[:, selected_feature_names.index(feat_col)]
        col_min, col_max = float(col.min()), float(col.max())
        centers = np.linspace(col_min, col_max, 3).tolist()
        sigma = max((col_max - col_min) / 3.0, 1e-3)
        mf_low = GaussianFuzzySet(mu=centers[0], sigma=sigma, term="low")
        mf_med = GaussianFuzzySet(mu=centers[1], sigma=sigma, term="medium")
        mf_high = GaussianFuzzySet(mu=centers[2], sigma=sigma, term="high")
        fs.add_linguistic_variable(
            fs_var,
            LinguisticVariable([mf_low, mf_med, mf_high], concept=FS_DISPLAY_NAMES[fs_var]),
        )
    fs.set_crisp_output_value("out", 0)
    grid_rules = [
        f"IF (ClumpThickness IS {ct}) AND (CellSize IS {cs}) AND (CellShape IS {csh}) THEN (out IS 0.5)"
        for ct, cs, csh in itertools.product(LINGUISTIC_TERMS, repeat=3)
    ]
    fs.add_rules(grid_rules)
    model = scikit_anfis(
        fs,
        description="WBCD_GridSearch_ANFIS_Inference",
        epoch=epochs,
        hybrid=True,
        label="c",
        zerotype=False,
    )
    return model


model = build_fs_and_model(X_train, split_cfg["epochs"])
model.load(str(model_path))
model.eval()
model.is_training = False

print("Loaded model:", model_path.name)
print("Num rules:", model.num_rules)
print("Epochs (train):", split_cfg["epochs"], "| LR:", split_cfg["learning_rate"])

 * Detected Sugeno model type
Loaded model: 20260628_172814_gridsearch_pca_feature_select_70_30_best_model.pkl
Num rules: 27
Epochs (train): 100 | LR: 0.05


In [21]:
def to_binary(pred):
    pred = np.asarray(pred).reshape(-1)
    return np.clip(np.round(pred), 0, 1).astype(int)


def predict_continuous(m, X):
    """Forward truc tiep — KHONG dung model.predict() (scikit-anfis shuffle batch lam lech thu tu)."""
    X_t = torch.from_numpy(np.asarray(X, dtype=np.float32)).float()
    m.eval()
    m.is_training = False
    with torch.no_grad():
        scores = m(X_t)
    return scores.numpy().reshape(-1)


def evaluate_split(name, y_true_arr, y_pred_arr, y_score_arr=None):
    acc = accuracy_score(y_true_arr, y_pred_arr)
    prec = precision_score(y_true_arr, y_pred_arr, zero_division=0)
    rec = recall_score(y_true_arr, y_pred_arr, zero_division=0)
    f1 = f1_score(y_true_arr, y_pred_arr, zero_division=0)
    try:
        score = y_score_arr if y_score_arr is not None else y_pred_arr
        auc = roc_auc_score(y_true_arr, score)
    except ValueError:
        auc = np.nan

    cm = confusion_matrix(y_true_arr, y_pred_arr, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        "split": name,
        "accuracy": float(acc),
        "precision": float(prec),
        "recall": float(rec),
        "f1_score": float(f1),
        "roc_auc": None if np.isnan(auc) else float(auc),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "support": int(len(y_true_arr)),
    }


# Predict toan bo 663 mau
y_score = predict_continuous(model, X_model)
y_pred = to_binary(y_score)

result_df = df_663[["sample_code_number", "class", "target", "target_label"]].copy()
result_df["y_pred_raw"] = y_score
result_df["y_pred"] = y_pred
result_df["pred_label"] = result_df["y_pred"].map({0: "benign", 1: "malignant"})
result_df["correct"] = (result_df["target"] == result_df["y_pred"]).astype(int)

full_metrics = evaluate_split("FULL_663", y_true, y_pred, y_score)

print(f"\n=== Metrics tren TOAN BO {USED_SAMPLES} mau ===")
for k, v in full_metrics.items():
    if k != "split":
        print(f"- {k}: {v}")

print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

print("\n5 mau dau:")
display(result_df.head())

print("\n5 mau du doan sai:")
display(result_df[result_df["correct"] == 0].head())


=== Metrics tren TOAN BO 663 mau ===
- accuracy: 0.9592760180995475
- precision: 0.944954128440367
- recall: 0.9321266968325792
- f1_score: 0.9384965831435079
- roc_auc: 0.9809227902786595
- tn: 430
- fp: 12
- fn: 15
- tp: 206
- support: 663

Confusion matrix:
[[430  12]
 [ 15 206]]

5 mau dau:


,sample_code_number,class,target,target_label,y_pred_raw,y_pred,pred_label,correct
0,1000025,2,0,benign,0.003101,0,benign,1
1,1002945,2,0,benign,0.620206,1,malignant,0
2,1015425,2,0,benign,0.010814,0,benign,1
3,1017023,2,0,benign,0.022547,0,benign,1
4,1017122,4,1,malignant,0.979445,1,malignant,1



5 mau du doan sai:


,sample_code_number,class,target,target_label,y_pred_raw,y_pred,pred_label,correct
1,1002945,2,0,benign,0.620206,1,malignant,0
11,1041801,4,1,malignant,0.304417,0,benign,0
19,1054590,4,1,malignant,0.379559,0,benign,0
23,1065726,4,1,malignant,0.264173,0,benign,0
48,1108449,4,1,malignant,0.304417,0,benign,0


In [22]:
# Luu ket qua predict
out_csv = models_dir / f"{STAMP}_{RUN_TAG}_{SPLIT_TAG}_full_663_predictions.csv"
metrics_out = models_dir / f"{STAMP}_{RUN_TAG}_{SPLIT_TAG}_full_663_metrics.json"

with open(metrics_out, "w", encoding="utf-8") as f:
    json.dump({"created_at": STAMP, "eval_scope": "FULL_663", "metrics": full_metrics}, f, indent=2, ensure_ascii=False)

result_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)
print("Saved:", metrics_out)
print("Tong mau:", len(result_df), "| Dung:", int(result_df["correct"].sum()), "| Sai:", int((result_df["correct"] == 0).sum()))

Saved: models\20260628_172814_gridsearch_pca_feature_select_70_30_full_663_predictions.csv
Saved: models\20260628_172814_gridsearch_pca_feature_select_70_30_full_663_metrics.json
Tong mau: 663 | Dung: 636 | Sai: 27
